In [1]:
# Figure out which sentences pre-handoff are pruned down to.
import json
data = []
with open('../data/exp_2_hsp_res.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [4]:
DATA = data[0]

In [8]:
DATA['first_essp_index']

72

In [10]:
# Steps up to handoff success
DATA['steps'][:DATA['first_essp_index']+1]

['Okay, so I need to figure out how many endpoints Figure 5 will have if we continue the pattern shown.',
 'Let me start by understanding the pattern from the given figures.',
 'First, let me look at Figure 1.',
 "It's a single line segment with three segments coming out of the top, forming a Y shape.",
 'Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).',
 "So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.",
 'So, the endpoints here would be the tips of these three lines.',
 'The bottom tip is at (0,-3), and the two upper tips are at (-2,2) and (2,2).',
 'So, Figure 1 has 3 endpoints.',
 'Now, moving to Figure 2.',
 'The Asymptote code draws a more complex figure.',
 'Let me try to visualize it.',
 'The description says each line-segment extremity is replaced by a gradually smaller Y.',
 'So, in Figure 1, each endp

In [9]:
# List of sentence indices to keep
keep_indices = [
    0,3,6,7,8,9,12,13,15,17,18,26,28,29,32,36,38
]
all_indices = list(range(40))
drop_indices = [num for num in all_indices if num not in keep_indices]
print(drop_indices)
pruned_steps = [DATA['steps'][idx] for idx in drop_indices]
pruned_steps

[1, 2, 4, 5, 10, 11, 14, 16, 19, 20, 21, 22, 23, 24, 25, 27, 30, 31, 33, 34, 35, 37, 39]


['Let me start by understanding the pattern from the given figures.',
 'First, let me look at Figure 1.',
 'Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).',
 "So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.",
 'The Asymptote code draws a more complex figure.',
 'Let me try to visualize it.',
 'Let me count the endpoints in Figure 2.',
 'Then there are lines from (4,1) to (5,0) to (6,1).',
 "Wait, maybe it's better to think recursively.",
 'Each endpoint from the previous figure is replaced by a Y.',
 'In Figure 1, there are 3 endpoints.',
 'Each of these endpoints is replaced by a Y in Figure 2.',
 'A Y has three endpoints, but one of them is connected to the previous segment.',
 'Wait, but when you replace an endpoint with a Y, how does that work?',
 'Let me think.',
 'Because the Y has two new branches.',
 'W

In [10]:
len(drop_indices)

23

In [10]:
len(keep_indices)

17 / 39

0.4358974358974359

In [11]:
problem = DATA['problem']
answer = DATA['answer']
pruned_steps

['Let me start by understanding the pattern from the given figures.',
 'First, let me look at Figure 1.',
 'Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).',
 "So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.",
 'The Asymptote code draws a more complex figure.',
 'Let me try to visualize it.',
 'Let me count the endpoints in Figure 2.',
 'Then there are lines from (4,1) to (5,0) to (6,1).',
 "Wait, maybe it's better to think recursively.",
 'Each endpoint from the previous figure is replaced by a Y.',
 'In Figure 1, there are 3 endpoints.',
 'Each of these endpoints is replaced by a Y in Figure 2.',
 'A Y has three endpoints, but one of them is connected to the previous segment.',
 'Wait, but when you replace an endpoint with a Y, how does that work?',
 'Let me think.',
 'Because the Y has two new branches.',
 'W

In [12]:
# Test if handoff succeeds.
import os
from vllm import LLM, SamplingParams

os.environ["CUDA_VISIBLE_DEVICES"] = "7"

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# Load the model
model = LLM(
    model="Qwen/Qwen3-4B",
    seed=9001,
    trust_remote_code=True,
    gpu_memory_utilization=0.9
)
tokenizer = model.get_tokenizer()

params_handoff = SamplingParams(
    n=20,
    temperature=0.7,
    top_p=0.95,
    max_tokens=31_000,
    seed=9001
)

# Set up the prompt
base_prompt = build_base_prompt(problem)

prompt_data = []

parital_reasoning = " ".join(pruned_steps)
prompt = f"{base_prompt}<think>\n{parital_reasoning}"
prompt_data.append(prompt)
    
generation_results = {}

prompts_to_generate = prompt_data

outputs = model.generate(
    prompts_to_generate, params_handoff, use_tqdm=True
)

# Run the model on the prompt 20 times

# Evaluate the proportion of correct final answers

/disk/u/gio/.conda/envs/liminal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 12-19 21:02:24 [utils.py:253] non-default args: {'trust_remote_code': True, 'seed': 9001, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 12-19 21:02:24 [model.py:637] Resolved architecture: Qwen3ForCausalLM
INFO 12-19 21:02:24 [model.py:1750] Using max model len 40960


2025-12-19 21:02:24,785	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.52it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:00<00:00,  2.87it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.10it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.11it/s]
(EngineCore_DP0 pid=3375277) 


(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:29 [default_loader.py:308] Loading weights took 1.52 seconds
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:29 [gpu_model_runner.py:3549] Model loading took 7.5552 GiB memory and 2.060010 seconds
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:35 [backends.py:655] Using cache directory: /disk/u/gio/.cache/vllm/torch_compile_cache/8043df2db3/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:35 [backends.py:715] Dynamo bytecode transform time: 5.95 s
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:38 [backends.py:216] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.244 s
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:42 [monitor.py:34] torch.compile takes 8.19 s in total
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:43 [gpu_worker.py:359] Available KV cache memory: 62.31 GiB
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:43 [kv_cache_utils.py:1286] GPU KV cache size: 453,696 tokens


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 30.58it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 37.76it/s]


(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:46 [gpu_model_runner.py:4466] Graph capturing finished in 3 secs, took 0.60 GiB
(EngineCore_DP0 pid=3375277) INFO 12-19 21:02:46 [core.py:254] init engine (profile, create kv cache, warmup model) took 17.10 seconds
INFO 12-19 21:02:47 [llm.py:343] Supported tasks: ['generate']


Processed prompts: 100%|██████████| 20/20 [02:48<00:00,  8.41s/it, est. speed input: 111.78 toks/s, output: 870.51 toks/s]


In [13]:
len(outputs)

1

In [14]:
output = outputs[0]

In [15]:
ground_truth = DATA['solution']

# Get generation results
completions = [o.text for o in output.outputs]

In [16]:
import re

class MathVerifier:
    """Answer extractor with normalization."""
    
    @staticmethod
    def extract_answer(text):
        if not text: return None
        # Handle cases where whitespace is inserted like \boxed { content }
        start_indices = [m.start() for m in re.finditer(r'\\boxed\s*\{', text)]
        if not start_indices: return None
        
        for start in start_indices:
            balance = 0
            # Find the first '{' after the \boxed
            content_start = text.find('{', start) + 1
            for i in range(content_start, len(text)):
                char = text[i]
                if char == '{': balance += 1
                elif char == '}':
                    if balance == 0: return text[content_start:i]
                    balance -= 1
        return None

    @staticmethod
    def extract_from_partial(text):
        """
        Extracts content when the prompt ends with \boxed{
        We look for the first closing brace '}' that isn't balanced by an opening '{'
        within the generated text itself.
        """
        if not text: return None
        balance = 0
        for i, char in enumerate(text):
            if char == '{':
                balance += 1
            elif char == '}':
                # If balance is 0, this '}' closes the ghost \boxed{ from the prompt
                if balance == 0:
                    return text[:i]
                balance -= 1
        # Fallback: if no closing brace found, return everything (model stopped early)
        return text

    @staticmethod
    def normalize_answer(text):
        if text is None: return ""
        text = text.strip().replace(" ", "")
        
        # --- FIX 1: Handle JSON-style double escaping ---
        # Turns \\left into \left, \\dfrac into \dfrac
        text = text.replace("\\\\", "\\") 
        
        # --- FIX 2: Standardize LaTeX fractions ---
        # Handle all fraction formats: \dfrac, \tfrac, dfrac (missing backslash)
        text = text.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")
        text = text.replace("dfrac", "frac")  # Handle cases where backslash was stripped
        
        # --- FIX 3: Remove sizing commands ---
        text = text.replace(r"\left", "").replace(r"\right", "")
        
        # --- FIX 4: Remove \text{...} wrappers ---
        text = re.sub(r'\\text\{(.*?)\}', r'\1', text)
        
        # --- FIX 5: Normalize plain fractions to LaTeX format ---
        # Convert "7/20" to "frac{7}{20}" for consistent comparison
        # This handles simple cases like "7/20", "12/5", etc.
        text = re.sub(r'(\d+)/(\d+)', r'frac{\1}{\2}', text)
        
        return text

    @staticmethod
    def is_correct(generated_text, ground_truth, is_partial=False):
        if is_partial:
            pred = MathVerifier.extract_from_partial(generated_text)
        else:
            pred = MathVerifier.extract_answer(generated_text)
            
        truth = MathVerifier.extract_answer(ground_truth)
        if truth is None: truth = ground_truth
        
        # Debug print to verify the fix works in your logs
        print()
        print(f"  Norm Pred: '{MathVerifier.normalize_answer(pred)}'")
        print(f"  Norm Truth: '{MathVerifier.normalize_answer(truth)}'")

        return MathVerifier.normalize_answer(pred) == MathVerifier.normalize_answer(truth)

correct_count = sum(
    1
    for comp in completions
    if MathVerifier.is_correct(comp, ground_truth)
)
accuracy = correct_count / 20
accuracy


  Norm Pred: '94'
  Norm Truth: '48'

  Norm Pred: '61'
  Norm Truth: '48'

  Norm Pred: '61'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '93'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '55'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '55'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '61'
  Norm Truth: '48'

  Norm Pred: '178'
  Norm Truth: '48'

  Norm Pred: '93'
  Norm Truth: '48'

  Norm Pred: '61'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '61'
  Norm Truth: '48'

  Norm Pred: '94'
  Norm Truth: '48'


0.3

In [17]:
# Steps up to handoff success
DATA['steps'][:DATA['first_essp_index']+1]

['Okay, so I need to figure out how many endpoints Figure 5 will have if we continue the pattern shown.',
 'Let me start by understanding the pattern from the given figures.',
 'First, let me look at Figure 1.',
 "It's a single line segment with three segments coming out of the top, forming a Y shape.",
 'Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).',
 "So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.",
 'So, the endpoints here would be the tips of these three lines.',
 'The bottom tip is at (0,-3), and the two upper tips are at (-2,2) and (2,2).',
 'So, Figure 1 has 3 endpoints.',
 'Now, moving to Figure 2.',
 'The Asymptote code draws a more complex figure.',
 'Let me try to visualize it.',
 'The description says each line-segment extremity is replaced by a gradually smaller Y.',
 'So, in Figure 1, each endp

In [6]:
len(DATA['steps'])

181

In [33]:
essp_keep_indices = [
    0,3,6,7,8,9,12,13,15,17,18,26,28,29,32,36,38,
    41, 43, 49, 56, 57, 58, 59, 65, 66, 72,
]
essp_pruned_steps = [DATA['steps'][idx] for idx in essp_keep_indices]
essp_pruned_steps
all_indices = list(range(73))

drop_indices = [num for num in all_indices if num not in essp_keep_indices]
print(drop_indices)
anti_pruned_steps = [DATA['steps'][idx] for idx in drop_indices]
essp_pruned_steps

un_pruned_steps = DATA['steps'][:73]
un_pruned_steps

[1, 2, 4, 5, 10, 11, 14, 16, 19, 20, 21, 22, 23, 24, 25, 27, 30, 31, 33, 34, 35, 37, 39, 40, 42, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 60, 61, 62, 63, 64, 67, 68, 69, 70, 71]


['Okay, so I need to figure out how many endpoints Figure 5 will have if we continue the pattern shown.',
 'Let me start by understanding the pattern from the given figures.',
 'First, let me look at Figure 1.',
 "It's a single line segment with three segments coming out of the top, forming a Y shape.",
 'Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).',
 "So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.",
 'So, the endpoints here would be the tips of these three lines.',
 'The bottom tip is at (0,-3), and the two upper tips are at (-2,2) and (2,2).',
 'So, Figure 1 has 3 endpoints.',
 'Now, moving to Figure 2.',
 'The Asymptote code draws a more complex figure.',
 'Let me try to visualize it.',
 'The description says each line-segment extremity is replaced by a gradually smaller Y.',
 'So, in Figure 1, each endp

In [ ]:
prompt = DATA['problem']

pruned_cot = ""
for step in anti_pruned_steps:
    pruned_cot += " " + step

pruned_cot += "\n</think>\nTherefore, the final answer is \\boxed{"

print(prompt)


If you continue this pattern in which each line-segment extremity is replaced by a gradually smaller Y in the next figure, in the manner shown, how many endpoints will Figure 5 have?

[asy]
draw((0,0)--(0,-3),linewidth(.75));
draw((0,0)--(-2,2),linewidth(.75));
draw((0,0)--(2,2),linewidth(.75));
label("Figure 1",(0,-3),S);

draw((5,0)--(5,-2),linewidth(.75));
draw((4,-3)--(5,-2)--(6,-3),linewidth(.75));
draw((4,1)--(5,0)--(6,1),linewidth(.75));
draw((3,1)--(4,1)--(4,2),linewidth(.75));
draw((6,2)--(6,1)--(7,1),linewidth(.75));
label("Figure 2",(5,-3),S);

draw((10,0)--(10,-2),linewidth(.75));
draw((9.5,-2.5)--(10,-2)--(10.5,-2.5),linewidth(.75));
draw((9,-2.5)--(9.5,-2.5)--(9.5,-3),linewidth(.75));
draw((11,-2.5)--(10.5,-2.5)--(10.5,-3),linewidth(.75));

draw((9,1)--(10,0)--(11,1),linewidth(.75));
draw((8.5,1)--(9,1)--(9,1.5),linewidth(.75));
draw((11.5,1)--(11,1)--(11,1.5),linewidth(.75));
draw((8.25,.75)--(8.5,1)--(8.25,1.25),linewidth(.75));
draw((8.75,1.75)--(9,1.5)--(9.25,1.75),li

In [44]:
del model

In [42]:
# Test if essp succeeds.
import os
from vllm import LLM, SamplingParams

problem = DATA['problem']
ground_truth = DATA['solution']

os.environ["CUDA_VISIBLE_DEVICES"] = "7"

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# Load the model
model = LLM(
    model="Qwen/Qwen3-4B",
    seed=9001,
    trust_remote_code=True,
    gpu_memory_utilization=0.9
)
tokenizer = model.get_tokenizer()

params_handoff = SamplingParams(
    n=20,
    temperature=0.7,
    top_p=0.95,
    max_tokens=5,
    seed=9001
)

# Set up the prompt
base_prompt = build_base_prompt(problem)

prompt_data = []

parital_reasoning = "\n".join(un_pruned_steps)
prompt = f"{base_prompt}<think>\n{parital_reasoning}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"
print(prompt)
prompt_data.append(prompt)
    
generation_results = {}

prompts_to_generate = prompt_data

outputs = model.generate(
    prompts_to_generate, params_handoff, use_tqdm=True
)

INFO 12-19 21:45:14 [utils.py:253] non-default args: {'trust_remote_code': True, 'seed': 9001, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 12-19 21:45:14 [model.py:631] Resolved architecture: Qwen3ForCausalLM
INFO 12-19 21:45:14 [model.py:1745] Using max model len 40960
INFO 12-19 21:45:14 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:15 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:16 [parallel_state.py:1208] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.201.16.108:57117 backend=nccl
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:16 [parallel_state.py:1394] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.65it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:00<00:00,  3.06it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.29it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.30it/s]
(EngineCore_DP0 pid=3382615) 


(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:19 [default_loader.py:314] Loading weights took 1.41 seconds
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:19 [gpu_model_runner.py:3338] Model loading took 7.5552 GiB memory and 2.176186 seconds
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:25 [backends.py:631] Using cache directory: /disk/u/gio/.cache/vllm/torch_compile_cache/5ef759b3b3/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:25 [backends.py:647] Dynamo bytecode transform time: 5.90 s
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:28 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.236 s
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:32 [monitor.py:34] torch.compile takes 8.14 s in total
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:33 [gpu_worker.py:359] Available KV cache memory: 62.31 GiB
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:33 [kv_cache_utils.py:1229] GPU KV cache size: 453,696 tokens


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 30.31it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 38.65it/s]


(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:37 [gpu_model_runner.py:4244] Graph capturing finished in 3 secs, took 0.60 GiB
(EngineCore_DP0 pid=3382615) INFO 12-19 21:45:37 [core.py:250] init engine (profile, create kv cache, warmup model) took 17.15 seconds
INFO 12-19 21:45:38 [llm.py:352] Supported tasks: ['generate']
<|im_start|>system
You are a helpful reasoning assistant. Output your final answer inside \boxed{}.<|im_end|>
<|im_start|>user
If you continue this pattern in which each line-segment extremity is replaced by a gradually smaller Y in the next figure, in the manner shown, how many endpoints will Figure 5 have?

[asy]
draw((0,0)--(0,-3),linewidth(.75));
draw((0,0)--(-2,2),linewidth(.75));
draw((0,0)--(2,2),linewidth(.75));
label("Figure 1",(0,-3),S);

draw((5,0)--(5,-2),linewidth(.75));
draw((4,-3)--(5,-2)--(6,-3),linewidth(.75));
draw((4,1)--(5,0)--(6,1),linewidth(.75));
draw((3,1)--(4,1)--(4,2),linewidth(.75));
draw((6,2)--(6,1)--(7,1),linewidth(.75));
label("Figure 2"

Processed prompts: 100%|██████████| 20/20 [00:00<00:00, 110.17it/s, est. speed input: 236892.15 toks/s, output: 556.29 toks/s]


In [43]:
import re

class MathVerifier:
    """Answer extractor with normalization."""
    
    @staticmethod
    def extract_answer(text):
        if not text: return None
        # Handle cases where whitespace is inserted like \boxed { content }
        start_indices = [m.start() for m in re.finditer(r'\\boxed\s*\{', text)]
        if not start_indices: return None
        
        for start in start_indices:
            balance = 0
            # Find the first '{' after the \boxed
            content_start = text.find('{', start) + 1
            for i in range(content_start, len(text)):
                char = text[i]
                if char == '{': balance += 1
                elif char == '}':
                    if balance == 0: return text[content_start:i]
                    balance -= 1
        return None

    @staticmethod
    def extract_from_partial(text):
        """
        Extracts content when the prompt ends with \boxed{
        We look for the first closing brace '}' that isn't balanced by an opening '{'
        within the generated text itself.
        """
        if not text: return None
        balance = 0
        for i, char in enumerate(text):
            if char == '{':
                balance += 1
            elif char == '}':
                # If balance is 0, this '}' closes the ghost \boxed{ from the prompt
                if balance == 0:
                    return text[:i]
                balance -= 1
        # Fallback: if no closing brace found, return everything (model stopped early)
        return text

    @staticmethod
    def normalize_answer(text):
        if text is None: return ""
        text = text.strip().replace(" ", "")
        
        # --- FIX 1: Handle JSON-style double escaping ---
        # Turns \\left into \left, \\dfrac into \dfrac
        text = text.replace("\\\\", "\\") 
        
        # --- FIX 2: Standardize LaTeX fractions ---
        # Handle all fraction formats: \dfrac, \tfrac, dfrac (missing backslash)
        text = text.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")
        text = text.replace("dfrac", "frac")  # Handle cases where backslash was stripped
        
        # --- FIX 3: Remove sizing commands ---
        text = text.replace(r"\left", "").replace(r"\right", "")
        
        # --- FIX 4: Remove \text{...} wrappers ---
        text = re.sub(r'\\text\{(.*?)\}', r'\1', text)
        
        # --- FIX 5: Normalize plain fractions to LaTeX format ---
        # Convert "7/20" to "frac{7}{20}" for consistent comparison
        # This handles simple cases like "7/20", "12/5", etc.
        text = re.sub(r'(\d+)/(\d+)', r'frac{\1}{\2}', text)
        
        return text

    @staticmethod
    def is_correct(generated_text, ground_truth, is_partial=False):
        if is_partial:
            pred = MathVerifier.extract_from_partial(generated_text)
        else:
            pred = MathVerifier.extract_answer(generated_text)
            
        truth = MathVerifier.extract_answer(ground_truth)
        if truth is None: truth = ground_truth
        
        # Debug print to verify the fix works in your logs
        print()
        print(f"  Norm Pred: '{MathVerifier.normalize_answer(pred)}'")
        print(f"  Norm Truth: '{MathVerifier.normalize_answer(truth)}'")

        return MathVerifier.normalize_answer(pred) == MathVerifier.normalize_answer(truth)

output = outputs[0]

# Get generation results
completions = [o.text for o in output.outputs]

correct_count = sum(
    1
    for comp in completions
    if MathVerifier.is_correct(comp, ground_truth, is_partial=True)
)
accuracy = correct_count / 20
accuracy


  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'

  Norm Pred: '3'
  Norm Truth: '48'


0.25

In [30]:
[o.text for o in output.outputs]

['3} for Figure ',
 '3} for Figure ',
 '6} for Figure ',
 '6}.\n\nWait,',
 '3 \\times 2',
 '32}.\n\nWait',
 '3} for Figure ',
 '32}.\n\nWait',
 '32}.\n\n**',
 '3} for Figure ',
 '6} for Figure ',
 '6} for Figure ',
 '32}.\n\n**',
 '3} for Figure ',
 '3} for Figure ',
 '3} for Figure ',
 '3 \\times 2',
 '6} for Figure ',
 '6} for Figure ',
 '3} for Figure ']